# Titanic Dataset

Most of the code cells below include comments explaining the task to be performed in those cells. Please delete the comments and add code to perform those tasks. There are a few code cells in which code has already been provided for you. In some cases, you will need to complete this code. 

# Import Statements

In [ ]:
# Importing libraries for data and making plots
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Loading scikit-learn for building and tuning ML models
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Importing joblib to save and load trained models
import joblib

# Ignoring warnings
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Load Training Data

In [ ]:
# Loading the training data 
train = pd.read_csv("/kaggle/input/titanic/train.csv")

In [ ]:
# Verifying the first few rows of the training data
train.head()

# Check for Missing Values

In [ ]:
# Checking how many missing values each column has
missing_values = train.isnull().sum()

# Showing the results to see which columns need cleaning
print("Missing values per column:")
print(missing_values)

# Check Label Distribution

In [ ]:
# Checking the label distribution to see how many passengers survived
label_distribution = train['Survived'].value_counts(normalize=True)

# Printing results
print(label_distribution)

# Preprocessing

In [ ]:
# Creating a new column named 'FamSize' to the DF, combining the siblings & spouses; also, the parents & children
train['FamSize'] = train['SibSp'] + train['Parch']

# Using the function below to determine the deck letter for each passenger
def set_deck(cabin):
    if str(cabin) == 'nan':
        return 'Missing'
    return cabin[0]

# Using the map()
train['Deck'] = train['Cabin'].map(set_deck)

# Checking the new column
print(train['Deck'].value_counts())

In [ ]:
# Creating the provided list of numerical feature names: 'Age', 'FamSize', 'Fare'
num_features = ['Age', 'FamSize', 'Fare']

# Creating a list of the provided categorical feature names: 'Sex', 'Pclass', 'Deck', 'Embarked'
cat_features = ['Sex', 'Pclass', 'Deck', 'Embarked']

# Combining the two previous lists into one list named 'features'
features = num_features + cat_features

# Creating a Pipeline object for processing the numerical features
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),   
    ('scaler', StandardScaler())                     
])

# Creating a Pipeline object for processing the categorical features
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),  
    ('encoder', OneHotEncoder(handle_unknown='ignore'))    
])

# Combining both pipelines into one ColunmTransformer
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

In [ ]:
# Fitting the preprocessor to the training data, using only the columns in the 'features' list
preprocessor.fit(train[features])

# Transforming the training data with the fitted preprocessor
X_train = preprocessor.transform(train[features])

# Creating a variable named y-train
y_train = train['Survived']

# Printing the shape of the results to make sure everything looks correct
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

# Model Selection

## Logistic Regression

In [ ]:
# Getting a Logistic Regression classifier
lr_clf = LogisticRegression(max_iter=5000, solver='saga', penalty='elasticnet', random_state=42)

# Defining a range of hyperparameters to test
lr_parameters = {
    'l1_ratio':[0, 0.5, 1],
    'C': [0.01, 0.1, 1, 10, 100]
}

# Setting up GridSearchCV for 10-fold cross-validation
lr_grid = GridSearchCV(lr_clf, lr_parameters, cv=10, refit='True', n_jobs=-1, verbose=10, scoring='accuracy')
lr_grid.fit(X_train, y_train)

# Fitting grid search to the training data
lr_grid.fit(X_train, y_train)

# Extracting the best model from grid search
lr_model = lr_grid.best_estimator_

# Displaying the results
print('Best Parameters:', lr_grid.best_params_)
print('Best CV Score:  ', lr_grid.best_score_)
print('Training Acc:   ', lr_model.score(X_train, y_train))

In [ ]:
# Run this cell without any changes to view the CV results.

lr_summary = pd.DataFrame(lr_grid.cv_results_['params'])
lr_summary['cv_score'] = lr_grid.cv_results_['mean_test_score']

for r in lr_parameters['l1_ratio']:
    temp = lr_summary.query(f'l1_ratio == {r}')
    plt.plot(temp.C, temp.cv_score, label=r)
plt.xscale('log')
plt.ylim([0.75, 0.82])
plt.xlabel('Regularization Parameter (C)')
plt.ylabel('CV Score')
plt.legend(title='L1 Ratio', loc='lower right')
plt.grid()
plt.show()

print(lr_summary.to_string(index=False))

## Decicion Trees

In [ ]:
# Getting a Decision Tree classifier
dt_clf = DecisionTreeClassifier(random_state=1)

# Defining the hyperparameter grid to test
dt_parameters = {
    'max_depth': [3, 5, 7, 9, None],
    'min_samples_leaf': [1, 2, 4, 6, 8]
}

# Setting up GridSearchCV for 10-fold cross-validation
dt_grid = GridSearchCV(dt_clf, dt_parameters, cv=10, refit='True', n_jobs=-1, verbose=0, scoring='accuracy')

# Fitting the grid search to the training data
dt_grid.fit(X_train, y_train)

# Getting the best model
dt_model = dt_grid.best_estimator_

print('Best Parameters:', dt_grid.best_params_)
print('Best CV Score:  ', dt_grid.best_score_)
print('Training Acc:   ', dt_model.score(X_train, y_train))

In [ ]:
# Run this cell without any changes to view the CV results.

dt_summary = pd.DataFrame(dt_grid.cv_results_['params'])
dt_summary['cv_score'] = dt_grid.cv_results_['mean_test_score']

for ms in dt_parameters['min_samples_leaf']:
    temp = dt_summary.query(f'min_samples_leaf == {ms}')
    plt.plot(temp.max_depth, temp.cv_score, label=ms)
plt.xlabel('Maximum Depth')
plt.ylabel('CV Score')
plt.legend(title='Min Samples')
plt.grid()
plt.show()

print(dt_summary.to_string(index=False))

## Random Forests

In [ ]:
%%time 

# Getting the Random Forest classifier
rf_clf = RandomForestClassifier(random_state=1, n_estimators=100)

# Defining the hyperparameter grid to test
rf_parameters = {
    'max_depth': [3, 5, 7, 9, None],
    'min_samples_leaf': [1, 2, 4, 6, 8]
}

# Setting up GridSearch CV for 10-fold cross-validation
rf_grid = GridSearchCV(rf_clf, rf_parameters, cv=10, refit='True', n_jobs=-1, verbose=0, scoring='accuracy')
rf_grid.fit(X_train, y_train)

# Fitting the grid search to the training data
rf_grid.fit(X_train, y_train)

# Getting the best model
rf_model = rf_grid.best_estimator_

print('Best Parameters:', rf_grid.best_params_)
print('Best CV Score:  ', rf_grid.best_score_)
print('Training Acc:   ', rf_model.score(X_train, y_train))

In [ ]:
# Run this cell without any changes to view the CV results.

rf_summary = pd.DataFrame(rf_grid.cv_results_['params'])
rf_summary['cv_score'] = rf_grid.cv_results_['mean_test_score']

for ms in rf_parameters['min_samples_leaf']:
    temp = rf_summary.query(f'min_samples_leaf == {ms}')
    plt.plot(temp.max_depth, temp.cv_score, label=ms)
plt.xlabel('Maximum Depth')
plt.ylabel('CV Score')
plt.legend(title='Min Samples')
plt.grid()
plt.show()

print(rf_summary.to_string(index=False))

# Save Pipeline and Model

In [ ]:
# Saving the preprocessing pipeline
joblib.dump(preprocessor, 'titanic_preprocessor.pkl')
print("Preprocessing pipeline saved as 'titanic_preprocessor.pkl'")

# Saving final model
final_model = rf_model

# Saving the trained model
joblib.dump(final_model, 'titanic_model.pkl')
print("Final model saved as 'titanic_model.pkl'")

In [ ]:
# Loading test data
test_df = pd.read_csv('/kaggle/input/titanic/test.csv')

# Recreating engineered features
test_df['FamSize'] = test_df['SibSp'] + test_df['Parch'] + 1
test_df['Deck'] = test_df['Cabin'].astype(str).str[0]

# Preprocessing data
test_features = preprocessor.transform(test_df)

# Getting predictions
predictions = final_model.predict(test_features)

# Creating a submission
submission = pd.read_csv('/kaggle/input/titanic/gender_submission.csv')

# Replacing 'Survived' column w/predictions
submission['Survived'] = predictions

# Saving the submission to CSV
submission.to_csv('submission.csv', index=False)
print("Submission file 'submission.csv' created!")